# Flash desktop gratis (noVNC)
Ejecutá las celdas en orden. Al final tendrás una URL pública trycloudflare.com para abrir el escritorio.

In [ ]:
!apt-get -y update
!apt-get -y install xfce4 xfce4-terminal novnc websockify x11vnc xvfb wget curl
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
%%bash
mkdir -p ~/apps/flashbrowser ~/Desktop
cd ~/apps/flashbrowser
wget -O FlashBrowser https://github.com/radubirsan/FlashBrowser/releases/download/v0.01/FlashBrowser_linux_0.01 || true
chmod +x FlashBrowser
cat > ~/Desktop/FlashBrowser.desktop <<'EOF'
[Desktop Entry]
Type=Application
Name=FlashBrowser
Exec=/root/apps/flashbrowser/FlashBrowser
Terminal=false
Categories=Game;
EOF
chmod +x ~/Desktop/FlashBrowser.desktop

In [ ]:
%%bash
export DISPLAY=:1
pkill -f x11vnc || true
pkill -f Xvfb || true
pkill -f websockify || true
Xvfb :1 -screen 0 1366x768x24 &
sleep 2
x11vnc -display :1 -nopw -forever -shared -rfbport 5900 &
websockify --web=/usr/share/novnc/ 6080 localhost:5900 &
startxfce4 &
nohup cloudflared tunnel --no-autoupdate --url http://localhost:6080 > /content/cloudflared.log 2>&1 &
sleep 5
grep -q trycloudflare /content/cloudflared.log || sleep 5
URL=$(grep -o 'https://[a-zA-Z0-9.-]*trycloudflare.com' /content/cloudflared.log | tail -n1)
echo "PUBLIC_URL=$URL"